# D1-09 Optional - Foreground and background uncertainty split

This optional notebook makes one controlled comparison for the carbon-fiber foreground from Day 1.

We re-import `bafu` as `bafu copy`, but set every imported exchange `uncertainty type` to `0` before writing the database. We then re-import the carbon-fiber foreground with active parameters as `carbon fiber copy`, link it to `bafu copy`, and compare Monte Carlo results against the original `carbon fiber` database.

## 1) Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bw2calc as bc
import bw2data as bd
import bw2io as bi

bd.projects.set_current('aalborg-rlcia-2026')

source_background = 'bafu'
source_foreground = 'carbon fiber'

background_copy = 'bafu copy'
foreground_copy = 'carbon fiber copy'
parameter_group_copy = 'exhaust_gas_treatment_2_copy'

method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')
biosphere_name = 'ecoinvent-3.10-biosphere'
activity_name = 'carbon fiber production, weaved, at factory'

bafu_file = Path('../../data/lci-bafu.xlsx')

carbon_fiber_file = Path('assets/lci-carbon-fiber.xlsx')

print('Current project:', bd.projects.current)
print('BAFU workbook:', bafu_file.resolve())
print('Carbon-fiber workbook:', carbon_fiber_file.resolve())
print('Biosphere database for imports:', biosphere_name)

## 2) Small importer helpers

In [ ]:
def rename_importer_database(importer, new_name):
    """
    This function renames the `database` field of every activity
    and change the `input` field of every exchange in every activity,
    to point to a new name.
    """
    old_name = importer.db_name
    for dataset in importer.data:
        old_dataset_database = dataset.get('database', old_name)
        dataset['database'] = new_name
        for exchange in dataset.get('exchanges', []):
            if exchange.get('database') in {old_name, old_dataset_database}:
                exchange['database'] = new_name
            if isinstance(exchange.get('input'), tuple) and exchange['input'][0] in {old_name, old_dataset_database}:
                exchange['input'] = (new_name, exchange['input'][1])
    importer.db_name = new_name

def set_all_exchange_uncertainty_types_to_zero(importer):
    """
    This function sets to zero whenever the field `uncertainty type`
    is present in an exchange.
    """
    count = 0
    for dataset in importer.data:
        for exchange in dataset.get('exchanges', []):
            exchange['uncertainty type'] = 0
            count += 1
    return count


def rename_activity_parameter_groups(importer, new_group):
    """
    This function renames a given parameter group, 
    to avoid collisions with a potential existing one.
    """
    count = 0
    for dataset in importer.data:
        for parameter in dataset.get('parameters', []):
            parameter['group'] = new_group
            count += 1
    return count

## 3) Re-import `bafu` without exchange uncertainty

In [ ]:
bafu_importer = bi.ExcelImporter(str(bafu_file))
bafu_importer.apply_strategies()

zeroed_exchanges = set_all_exchange_uncertainty_types_to_zero(bafu_importer)
rename_importer_database(bafu_importer, background_copy)

bafu_importer.match_database(fields=('name', 'reference product', 'location'))
bafu_importer.match_database(
    biosphere_name,
    fields=('name', 'unit', 'categories'),
    relink=True,
    edge_kinds=['biosphere'],
)
print('BAFU copy import statistics:')
bafu_importer.statistics()

bafu_importer.write_database(db_name=background_copy, activate_parameters=False)
print('Imported exchanges with uncertainty type set to 0:', zeroed_exchanges)

In [ ]:
bd.databases

## 4) Re-import carbon fiber with parameters and link it to `bafu copy`

In [ ]:
migration_name = 'carbon-fiber-demo-fixes'

carbon_importer = bi.ExcelImporter(str(carbon_fiber_file))
carbon_importer.apply_strategies()
carbon_importer.data = bi.strategies.migrate_exchanges(db=carbon_importer.data, migration=migration_name)

rename_importer_database(carbon_importer, foreground_copy)
renamed_parameters = rename_activity_parameter_groups(carbon_importer, parameter_group_copy)

carbon_importer.match_database(fields=('name', 'reference product', 'location', 'unit'))

# here, we link against the copy of the `bafu` database
carbon_importer.match_database(background_copy, fields=('name', 'reference product', 'location', 'unit'))

carbon_importer.match_database(
    biosphere_name,
    fields=('name', 'unit', 'categories'),
    relink=True,
    edge_kinds=['biosphere'],
)
print('Carbon-fiber copy import statistics:')
carbon_importer.statistics()

if list(carbon_importer.unlinked):
    raise RuntimeError('Carbon-fiber copy still has unlinked exchanges; inspect carbon_importer.unlinked before writing.')

carbon_importer.write_database(
    db_name=foreground_copy,
    activate_parameters=True
)
print('Imported activity parameters:', renamed_parameters)

## 5) Select the same carbon-fiber activity in both foreground databases

In [ ]:
def deterministic_score(activity):
    lca = bc.LCA({activity: 1}, method=method)
    lca.lci()
    lca.lcia()
    return float(lca.score)


original_activity = [
    act for act in bd.Database("carbon fiber")
    if "weaved, at factory" in act["name"]
][0]

copied_activity = [
    act for act in bd.Database("carbon fiber copy")
    if "weaved, at factory" in act["name"]
][0]

activity_table = pd.DataFrame([
    {
        'label': 'original foreground linked to original bafu',
        'database': original_activity['database'],
        'activity': original_activity['name'],
        'location': original_activity['location'],
        'unit': original_activity['unit'],
    },
    {
        'label': 're-imported foreground linked to bafu copy',
        'database': copied_activity['database'],
        'activity': copied_activity['name'],
        'location': copied_activity['location'],
        'unit': copied_activity['unit'],
    },
])

deterministic_scores = activity_table.copy()
deterministic_scores['deterministic score'] = [
    deterministic_score(original_activity),
    deterministic_score(copied_activity),
]

score_ratio = deterministic_scores.loc[1, 'deterministic score'] / deterministic_scores.loc[0, 'deterministic score']
if score_ratio < 0.1 or score_ratio > 10:
    raise RuntimeError(
        'The copied foreground score is far from the original. '
        'Check that bafu copy biosphere exchanges are linked to the same biosphere database used by the method.'
    )

deterministic_scores

## 6) Run Monte Carlo and plot the results

In [ ]:
def monte_carlo_scores(activity, iterations=500, seed=2026):
    lca = bc.LCA(
        {activity: 1},
        method=method,
        use_distributions=True,
        seed_override=seed,
    )
    lca.lci()
    lca.lcia()

    scores = [float(lca.score)]
    for _ in range(iterations - 1):
        next(lca)
        scores.append(float(lca.score))
    return np.array(scores)


iterations = 500
seed = 2026

scores = pd.DataFrame({
    'foreground + background uncertainty': monte_carlo_scores(original_activity, iterations, seed),
    'foreground uncertainty only': monte_carlo_scores(copied_activity, iterations, seed),
})

summary = scores.describe(percentiles=[0.05, 0.5, 0.95]).T[
    ['mean', 'std', '5%', '50%', '95%']
]
summary['p95 - p05'] = summary['95%'] - summary['5%']
summary

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(
    scores['foreground + background uncertainty'],
    bins=100,
    alpha=0.6,
    label='foreground + background uncertainty',
)
plt.hist(
    scores['foreground uncertainty only'],
    bins=100,
    alpha=0.6,
    label='foreground uncertainty only',
)
plt.xlabel(f"Climate score [{bd.Method(method).metadata.get('unit', 'impact units')}]")
plt.ylabel('Frequency')
plt.title('Carbon fiber production Monte Carlo')
plt.legend()
plt.tight_layout()
plt.show()

## Interpretation

The original `carbon fiber` database samples uncertainty from both the foreground and the linked `bafu` background.

The re-imported `carbon fiber copy` database is linked to `bafu copy`, where each imported BAFU exchange has `uncertainty type = 0`. The difference between the two histograms shows the effect of removing BAFU background exchange uncertainty.